# Zac Stritch-Hoddle PCA Analysis of OpenPose Data.

## Imports and helper functions.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def load_data(file_path, norm=None, detren=True, interpolate=True):
    """
    Load time-series pose data from a CSV file.
    The file should have column names for each keypoint (e.g., nose_x, nose_y).
    If norm is zscore, the data will be zscore normalized.
    if norm is unit, the data will be 0-1 normalized.
    """
    # load data
    data = pd.read_csv(file_path)

    # interpolate missing values
    if interpolate:
        data = interpolate_nans(data)

    # linear detrend
    if detren:
        data = linear_detrend(data)

    # normalize data
    if norm == 'zscore':
        return normalize_data(data, norm='zscore')
    elif norm == 'uint':
        return normalize_data(data, norm='uint')
    else:
        return data
    
def normalize_data(data, norm=None):
    """
    Normalize each column of data using z-score or min-max scaling.

    Parameters:
    - data (pd.DataFrame): The data to normalize.
    - norm (str): The normalization method. Options:
        - 'zscore': Normalize using z-score ((x - mean) / std).
        - 'unit': Normalize to the range [0, 1] ((x - min) / (max - min)).
        - None: No normalization.

    Returns:
    - pd.DataFrame: The normalized data.
    """
    if norm == 'zscore':
        return (data - data.mean()) / data.std()
    elif norm == 'uint':  # Corrected typo from "uint" to "unit"
        return (data - data.min()) / (data.max() - data.min())
    return data

def linear_detrend(data):
    """
    Linearly detrend each column of data.

    Parameters:
    - data (pd.DataFrame): The data to detrend.

    Returns:
    - pd.DataFrame: The detrended data.
    """
    return data.apply(lambda x: x - np.polyval(np.polyfit(data.index, x, 1), data.index))

def interpolate_nans(data):
    """
    Linearly interpolate NaN values in a DataFrame or Series.

    Parameters:
    - data (pd.DataFrame or pd.Series): The data with potential NaN values.

    Returns:
    - pd.DataFrame or pd.Series: Data with NaN values linearly interpolated.
    """
    if isinstance(data, pd.DataFrame):
        return data.interpolate(method='linear', axis=0, limit_direction='both')
    elif isinstance(data, pd.Series):
        return data.interpolate(method='linear', limit_direction='both')
    else:
        raise TypeError("Input must be a pandas DataFrame or Series.")
    
def perform_pca(data, selected_keypoints, n_components=4):
    """
    Perform PCA on the selected keypoints.
    """
    selected_data = data[selected_keypoints]
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(selected_data)

    return pca, principal_components

def cross_correlation(data, pca_ts, selected_keypoints):
    """
    Computes the cross-correlation between the PCA time series (PC1, PC2)
    and the original data time series for the selected keypoints.

    Parameters:
    - data (pd.DataFrame): Original time series data for the keypoints.
    - pca_ts (np.ndarray): Principal component time series (2D array from PCA).
    - selected_keypoints (list): List of keypoints to compare with PCA time series.

    Returns:
    - correlation_results (pd.DataFrame): DataFrame with correlation values for PC1 and PC2.
    """
    # Check that all selected keypoints exist in the data
    missing_keypoints = [key for key in selected_keypoints if key not in data.columns]
    if missing_keypoints:
        raise KeyError(f"The following keypoints are missing in the data: {missing_keypoints}")

    # Initialize a dictionary to store correlations
    correlation_results = {"Keypoint": [], "PC1_Correlation": [], "PC2_Correlation": []}

    # Iterate through each keypoint
    for keypoint in selected_keypoints:
        try:
            # Cross-correlation with PC1
            correlation_pc1 = np.corrcoef(data[keypoint], pca_ts[:, 0])[0, 1]

            # Cross-correlation with PC2
            correlation_pc2 = np.corrcoef(data[keypoint], pca_ts[:, 1])[0, 1]

            # Append results
            correlation_results["Keypoint"].append(keypoint)
            correlation_results["PC1_Correlation"].append(correlation_pc1)
            correlation_results["PC2_Correlation"].append(correlation_pc2)
        except Exception as e:
            print(f"Error processing keypoint {keypoint}: {e}")
            correlation_results["Keypoint"].append(keypoint)
            correlation_results["PC1_Correlation"].append(None)
            correlation_results["PC2_Correlation"].append(None)

    # Convert to DataFrame for better readability
    correlation_results_df = pd.DataFrame(correlation_results)
    return correlation_results_df

def plot_time_series(data, selected_keypoints, principal_components, pnum_components=3, title="Time-Series Data"):
    """
    Plot the time-series data and the first two principal components.
    """
    time = np.arange(data.shape[0])

    # Plot the time series of the selected keypoints
    plt.figure(figsize=(10, 5))
    for kp in selected_keypoints:
        plt.plot(time, data[kp], label=f"{kp}", linestyle=":", alpha=0.4)

    # Plot the time series of the first and second principal components
    for i in range(pnum_components):
        plt.plot(time, principal_components[:, i], label=f"PC{i+1}", linestyle="-")

    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title(title)
    plt.legend(["PC1", "PC2", "PC3"])  
    plt.grid(True)
    plt.show()

def plot_acc_time_series(principal_components, pnum_components=3, title="PCA ACC Data"):
    """
    Plot the time-series data and the first two principal components.
    """
    time = np.arange(principal_components.shape[0])

    # Plot the time series of the selected keypoints
    plt.figure(figsize=(10, 5))

    # convert to acceleration
    acceleration = np.diff(np.diff(principal_components, axis=0, prepend=0), axis=0, prepend=0)

    # Plot the time series of the first and second principal components
    for i in range(pnum_components):
        plt.plot(time, acceleration[:, i], label=f"PC{i+1}", linestyle="-")

    plt.xlabel("Time")
    plt.ylabel("ACC")
    plt.title(title)
    plt.legend(["PC1", "PC2", "PC3"])  
    plt.grid(True)
    plt.show()

## Load and set keypoint lists for analysis.

In [8]:
def extract_keypoints(file_path_or_data, sets=["face"]):
    """
    Extract keypoints from the dataset based on the specified sets.

    Parameters:
    - file_path_or_data (str or pd.DataFrame): Path to the CSV file or a DataFrame.
    - sets (list): List of sets to include in the output.

    Returns:
    - dict: Dictionary with keys as set names and values as lists of relevant column headers.
    """
    # Load the data if a file path is provided
    if isinstance(file_path_or_data, str):
        data = pd.read_csv(file_path_or_data)
    elif isinstance(file_path_or_data, pd.DataFrame):
        data = file_path_or_data
    else:
        raise ValueError("Input must be a file path or a pandas DataFrame.")

    # Get headers
    headers = data.columns.tolist()

    # Generate face keypoints (grouping everything as "face")
    face_keypoints = [
        header for header in headers if any(
            face_label in header.lower() for face_label in [
                "face", "eye", "pupil", "magnitude"
            ]
        )
    ]

    # Create a dictionary to store results
    keypoints_dict = {}
    if "face" in sets:
        keypoints_dict["face"] = face_keypoints

    return keypoints_dict

## Running PCA analysis with three principal components.

### This runs a PCA analysis of the OpenPose keypoint data using three principal components. The explained variance ratios, as well as the RMS for these components output to 'exp3_PCA_data.csv'

### The first and second principal component weightings for each participant's keypoints were output to 'PCA_weightings.csv'
### The first and second principal component time-series were output to 'PCA_time_series.csv'

In [ ]:
import os
import pandas as pd
import numpy as np

# Define paths
data_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Exp3_FaceEyeData'
output_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA'
output_csv = os.path.join(output_path, 'exp3_PCA_data.csv')
pca_weightings_csv = os.path.join(output_path, 'PCA_weightings.csv')
pca_time_series_csv = os.path.join(output_path, 'PCA_time_series.csv')

# Initialize lists to store data
all_trial_stats = []
pca_weightings_data = []
pca_time_series_data = []

# Iterate through files in the directory
for file_name in os.listdir(data_path):
    if file_name.endswith('.csv') and "_" in file_name:
        # Extract participantID, trialNumber, and difficultyCondition
        parts = file_name.split("_")
        participantID = parts[0]
        trialNumber = parts[1]
        difficultyCondition = parts[2].split(".")[0]  # Remove the file extension

        # Construct file path
        file_path = os.path.join(data_path, file_name)

        # Load the data
        try:
            pose_data = load_data(file_path, norm=None)  # Ensure load_data function is defined
        except FileNotFoundError:
            print(f"File not found: {file_path}")
            continue

        # Extract keypoints
        keypoints = extract_keypoints(pose_data, sets=["face"])  # Ensure extract_keypoints is defined
        keypoints = [key for sublist in keypoints.values() for key in sublist]

        # Centre the data prior to analysis
        pose_data = pose_data[keypoints] - pose_data[keypoints].mean(axis=0)

        # Perform PCA
        pca, principal_components = perform_pca(pose_data, keypoints, n_components=6)

        # Save PCA weightings
        weightings = pca.components_.T  # Transpose to match keypoints to components
        weightings_row = {"Trial": f"{participantID}_{trialNumber}_{difficultyCondition}"}
        weightings_row.update({keypoints[i]: weightings[i, 0] for i in range(len(keypoints))})  # PCA-C1
        pca_weightings_data.append(weightings_row)

        # Save PCA time series (first and second principal components)
        for time_idx, (pc1, pc2) in enumerate(zip(principal_components[:, 0], principal_components[:, 1])):
            pca_time_series_data.append({
                "Trial": f"{participantID}_{trialNumber}_{difficultyCondition}",
                "Time": time_idx,
                "PC1": pc1,
                "PC2": pc2
            })

        # Calculate RMS
        rms = np.sqrt(np.mean(principal_components**2, axis=0))

        # Save trial stats
        trial_stats = {
            "ParticipantID": participantID,
            "Trial Number": trialNumber,
            "Difficulty Condition": difficultyCondition,
            "PCA-C1 EVR": pca.explained_variance_ratio_[0],
            "PCA-C2 EVR": pca.explained_variance_ratio_[1],
            "PCA-C3 EVR": pca.explained_variance_ratio_[2],
            "PCA-C1 RMS": rms[0],
            "PCA-C2 RMS": rms[1],
            "PCA-C3 RMS": rms[2],
        }
        all_trial_stats.append(trial_stats)

# Convert all trial stats to a DataFrame and save to CSV
df_stats = pd.DataFrame(all_trial_stats)
df_stats.to_csv(output_csv, index=False)

# Convert PCA weightings to a DataFrame and save to CSV
df_weightings = pd.DataFrame(pca_weightings_data)
df_weightings.to_csv(pca_weightings_csv, index=False)

# Convert PCA time series to a DataFrame and save to CSV
df_time_series = pd.DataFrame(pca_time_series_data)
df_time_series.to_csv(pca_time_series_csv, index=False)

print(f"Analysis complete. Data saved to {output_csv}")
print(f"PCA weightings saved to {pca_weightings_csv}")
print(f"PCA time series saved to {pca_time_series_csv}")
